In [2]:
# imports and settings

import os
import gpxpy
import time
import torch
import librosa
import pickle
import warnings
import pandas as pd
import networkx as nx

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

from scipy import signal
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import find_peaks, butter, filtfilt, welch, get_window
from scipy.ndimage import gaussian_filter
from scipy.io import wavfile
from scipy.stats import wasserstein_distance_nd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.io import write_image

# do not show warnings
warnings.filterwarnings("ignore")

# autoreload for development
%load_ext autoreload
%autoreload 2

print("Imports complete.")

Imports complete.


### 1. IQ Demodulation (Digital Down Conversion)
This is the industry standard for this exact problem. It is highly resistant to out-of-band noise and provides a continuous phase stream with very low computational overhead, making it ideal for localized, offline architectures.

**The Math:**
Given your incoming signal $x(t) = A(t)\cos(\omega_c t + \phi(t)) + n(t)$, where $\omega_c$ is your known carrier frequency and $\phi(t)$ is the changing phase:

1.  **Mix with Local Oscillator (LO):** Multiply the audio stream by a complex exponential at the known frequency.
    $$y(t) = x(t) \cdot e^{-j\omega_c t}$$
    Practically, this means generating two parallel streams:
    $$I_{raw}(t) = x(t) \cdot \cos(\omega_c t)$$
    $$Q_{raw}(t) = x(t) \cdot (-\sin(\omega_c t))$$
2.  **Low-Pass Filter (LPF):** The mixing process creates components at baseband (your phase signal) and at $2\omega_c$. Apply a low-pass filter (like a moving average, FIR, or CIC filter) to remove the high-frequency term.
    $$I(t) \approx \frac{A(t)}{2} \cos(\phi(t))$$
    $$Q(t) \approx \frac{A(t)}{2} \sin(\phi(t))$$
3.  **Extract Phase:** Calculate the four-quadrant inverse tangent.
    $$\phi(t) = \text{arctan2}(Q(t), I(t))$$

**Why it's robust:** The low-pass filter bandwidth explicitly defines your "short time interval" (integration time). A narrower filter yields higher Signal-to-Noise Ratio (SNR) and stability, while a wider filter allows you to track faster phase changes. 

In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt

def estimate_phase_iq(signal: np.ndarray, f_carrier: float, fs: float, cutoff_hz: float) -> np.ndarray:
    """
    Extracts the continuous phase of a signal using IQ demodulation.
    
    Args:
        signal: 1D numpy array containing the audio time series.
        f_carrier: The known center frequency in Hz.
        fs: The sampling rate in Hz.
        cutoff_hz: Low-pass filter cutoff frequency. Determines the "short time interval" integration.
                   Should be set to the maximum expected rate of change of the phase.
                   
    Returns:
        1D numpy array of the unwrapped phase estimate.
    """
    t = np.arange(len(signal)) / fs
    
    # 1. Mix with Local Oscillator
    i_raw = signal * np.cos(2 * np.pi * f_carrier * t)
    q_raw = signal * -np.sin(2 * np.pi * f_carrier * t)
    
    # 2. Low-Pass Filter
    nyquist = 0.5 * fs
    normal_cutoff = cutoff_hz / nyquist
    b, a = butter(N=4, Wn=normal_cutoff, btype='low', analog=False)
    
    # Using filtfilt for zero-phase distortion
    i_filtered = filtfilt(b, a, i_raw)
    q_filtered = filtfilt(b, a, q_raw)
    
    # 3. Extract and Unwrap Phase
    phase_wrapped = np.arctan2(q_filtered, i_filtered)
    phase_unwrapped = np.unwrap(phase_wrapped)
    
    return phase_unwrapped

### 2. The Goertzel Algorithm (Block-based)
If you do not need a continuous sample-by-sample phase stream and instead want a single phase estimate for every short, discrete block of audio (e.g., every 10ms), the Goertzel algorithm is exceptionally efficient.

* **Mechanism:** It computes a single bin of the Discrete Fourier Transform (DFT).
* **Advantage:** It is significantly faster than computing an FFT or STFT if you only care about one specific frequency target. 
* **Output:** It yields a complex number $X[k]$ for that block. The phase is simply $\text{arctan2}(\text{Im}(X[k]), \text{Re}(X[k]))$.
* **Limitation:** It is vulnerable to spectral leakage if the block length is not an integer multiple of the signal period. Applying a window function (like Hanning) to the short interval before running Goertzel mitigates this but widens the main lobe.

In [ ]:
def estimate_phase_goertzel(block: np.ndarray, f_target: float, fs: float) -> float:
    """
    Computes a single phase estimate for a specific block of audio using the Goertzel algorithm.
    
    Args:
        block: 1D numpy array of audio samples (the short time interval).
        f_target: The known target frequency in Hz.
        fs: The sampling rate in Hz.
        
    Returns:
        A single float representing the wrapped phase (-pi to pi) for this block.
    """
    N = len(block)
    
    # Calculate the generalized bin index (k does not need to be an integer)
    k = N * (f_target / fs)
    omega = (2 * np.pi * k) / N
    
    cosine = np.cos(omega)
    sine = np.sin(omega)
    coeff = 2 * cosine
    
    s_prev = 0.0
    s_prev2 = 0.0
    
    # IIR filter loop
    for x in block:
        s = x + coeff * s_prev - s_prev2
        s_prev2 = s_prev
        s_prev = s
        
    # Calculate the complex DFT value for the target bin
    real_part = s_prev - s_prev2 * cosine
    imag_part = s_prev2 * sine
    
    # Extract phase
    phase = np.arctan2(imag_part, real_part)
    
    return phase

### 3. Software Phase-Locked Loop (PLL)
If your "known frequency" might experience slight Doppler shifts, hardware clock drift, or if the signal drops out momentarily, a software PLL provides the ultimate stability.

* **Mechanism:** A PLL uses a phase detector to measure the error between the incoming signal and a Numerically Controlled Oscillator (NCO). This error is fed through a loop filter (typically a Proportional-Integral controller) to adjust the NCO's phase and frequency to lock onto the input.
* **Advantage:** The loop filter acts as a flywheel. If the audio signal gets swamped by a sudden transient noise or a brief dropout, the PLL maintains a stable phase trajectory based on its prior state. 
* **Implementation Note:** PLLs require careful tuning of the loop filter coefficients (damping factor and natural frequency) to balance responsiveness to phase changes against noise jitter.

In [ ]:
def estimate_phase_pll(signal: np.ndarray, f_center: float, fs: float, loop_bw_hz: float) -> np.ndarray:
    """
    Tracks the phase of a signal using a software Phase-Locked Loop (PLL).
    
    Args:
        signal: 1D numpy array containing the audio time series.
        f_center: The expected center/carrier frequency in Hz.
        fs: The sampling rate in Hz.
        loop_bw_hz: The bandwidth of the loop filter. A narrower bandwidth 
                    rejects more noise but tracks phase changes more slowly.
                    
    Returns:
        1D numpy array of the unwrapped relative phase.
    """
    N = len(signal)
    nco_phase_array = np.zeros(N)
    
    # Numerically Controlled Oscillator (NCO) initial state
    nco_phase = 0.0
    nco_freq_center = 2 * np.pi * f_center / fs
    nco_freq = nco_freq_center
    
    # Proportional-Integral (PI) Loop Filter gains
    # Standard damping factor (zeta) of 1 / sqrt(2) for optimal settling
    zeta = 0.707 
    wn = 2 * np.pi * loop_bw_hz / fs
    
    K_p = 2 * zeta * wn
    K_i = wn**2
    
    integrator = 0.0
    
    for i in range(N):
        # 1. Phase Detector (Multiplier)
        # We multiply the incoming signal by the NCO's quadrature (-sine) component.
        # If the input is A*cos(wt + phi), driving the error to 0 locks the NCO phase.
        nco_q = -np.sin(nco_phase)
        phase_error = signal[i] * nco_q
        
        # 2. Loop Filter (PI Controller)
        integrator += K_i * phase_error
        freq_update = nco_freq_center + K_p * phase_error + integrator
        
        # 3. Update NCO
        nco_phase += freq_update
        nco_phase_array[i] = nco_phase

    # The NCO phase tracks the absolute total phase (wt + phi). 
    # To get just the changing relative phase phi(t), we subtract the ideal linear carrier phase.
    ideal_carrier_phase = nco_freq_center * np.arange(N)
    relative_phase = nco_phase_array - ideal_carrier_phase
    
    return np.unwrap(relative_phase)